In [6]:
import sqlite3
import os
from datetime import datetime, timedelta

# ================= RESET DATABASE (IMPORTANT FIX) =================
if os.path.exists("library.db"):
    os.remove("library.db")

# ================= CONNECTION =================
conn = sqlite3.connect("library.db")
cursor = conn.cursor()

# ================= TABLES =================

cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    author TEXT,
    category TEXT,
    status TEXT DEFAULT 'Available'
)
""")

cursor.execute("""
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    role TEXT,
    department TEXT
)
""")

cursor.execute("""
CREATE TABLE issue_records (
    issue_id INTEGER PRIMARY KEY AUTOINCREMENT,
    book_id INTEGER,
    member_id INTEGER,
    issue_date TEXT,
    return_date TEXT,
    status TEXT
)
""")

cursor.execute("""
CREATE TABLE fines (
    fine_id INTEGER PRIMARY KEY AUTOINCREMENT,
    member_id INTEGER,
    amount REAL,
    reason TEXT
)
""")

conn.commit()

# ================= BOOK MODULE =================

def add_book(title, author, category):
    cursor.execute("INSERT INTO books(title, author, category) VALUES (?,?,?)",
                   (title, author, category))
    conn.commit()
    print("✔ Book Added")

def view_books():
    cursor.execute("SELECT * FROM books")
    for row in cursor.fetchall():
        print(row)

def search_book(keyword):
    cursor.execute("""
    SELECT * FROM books WHERE title LIKE ? OR author LIKE ?
    """, ('%'+keyword+'%', '%'+keyword+'%'))
    return cursor.fetchall()

def delete_book(book_id):
    cursor.execute("DELETE FROM books WHERE book_id=?", (book_id,))
    conn.commit()
    print("✔ Book Deleted")

# ================= MEMBER MODULE =================

def add_member(name, role, department):
    cursor.execute("INSERT INTO members(name, role, department) VALUES (?,?,?)",
                   (name, role, department))
    conn.commit()
    print("✔ Member Added")

def view_members():
    cursor.execute("SELECT * FROM members")
    for row in cursor.fetchall():
        print(row)

# ================= ISSUE MODULE =================

def issue_book(book_id, member_id):
    issue_date = datetime.now()
    return_date = issue_date + timedelta(days=7)

    cursor.execute("""
    INSERT INTO issue_records(book_id, member_id, issue_date, return_date, status)
    VALUES (?,?,?,?,?)
    """, (book_id, member_id, str(issue_date), str(return_date), "Issued"))

    cursor.execute("UPDATE books SET status='Issued' WHERE book_id=?", (book_id,))
    conn.commit()
    print("✔ Book Issued")

def return_book(issue_id):
    cursor.execute("SELECT book_id FROM issue_records WHERE issue_id=?", (issue_id,))
    data = cursor.fetchone()

    if data:
        book_id = data[0]

        cursor.execute("UPDATE issue_records SET status='Returned' WHERE issue_id=?",
                       (issue_id,))

        cursor.execute("UPDATE books SET status='Available' WHERE book_id=?",
                       (book_id,))

        conn.commit()
        print("✔ Book Returned")

# ================= FINE SYSTEM =================

def calculate_fine(issue_id):
    cursor.execute("SELECT return_date FROM issue_records WHERE issue_id=?",
                   (issue_id,))
    data = cursor.fetchone()

    if data:
        return_date = datetime.fromisoformat(data[0])

        if datetime.now() > return_date:
            days = (datetime.now() - return_date).days
            fine = days * 10

            cursor.execute("""
            INSERT INTO fines(member_id, amount, reason)
            SELECT member_id, ?, 'Late Return'
            FROM issue_records WHERE issue_id=?
            """, (fine, issue_id))

            conn.commit()
            print("💰 Fine:", fine)
            return fine

    print("✔ No Fine")

# ================= REPORTS =================

def system_stats():
    cursor.execute("SELECT COUNT(*) FROM books")
    b = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM members")
    m = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM issue_records")
    i = cursor.fetchone()[0]

    print("\n📊 SYSTEM STATS")
    print("Books:", b)
    print("Members:", m)
    print("Issued:", i)

# ================= DEMO DATA (3-MEMBER PROJECT) =================

print("\n🚀 SYSTEM STARTED\n")

add_member("Ali", "Student", "CS")
add_member("Ahmed", "Student", "IT")
add_member("Sara", "Librarian", "Admin")

add_book("Python", "John", "Programming")
add_book("DBMS", "Khan", "Database")
add_book("AI", "Smith", "CS")

print("\n📚 BOOKS:")
view_books()

print("\n👥 MEMBERS:")
view_members()

issue_book(1, 1)
issue_book(2, 2)

return_book(1)

calculate_fine(1)

system_stats()

print("\n✅ PROJECT COMPLETED SUCCESSFULLY")


🚀 SYSTEM STARTED

✔ Member Added
✔ Member Added
✔ Member Added
✔ Book Added
✔ Book Added
✔ Book Added

📚 BOOKS:
(1, 'Python', 'John', 'Programming', 'Available')
(2, 'DBMS', 'Khan', 'Database', 'Available')
(3, 'AI', 'Smith', 'CS', 'Available')

👥 MEMBERS:
(1, 'Ali', 'Student', 'CS')
(2, 'Ahmed', 'Student', 'IT')
(3, 'Sara', 'Librarian', 'Admin')
✔ Book Issued
✔ Book Issued
✔ Book Returned
✔ No Fine

📊 SYSTEM STATS
Books: 3
Members: 3
Issued: 2

✅ PROJECT COMPLETED SUCCESSFULLY
